# 02-4. 多変数関数と偏微分 — 動かして確かめる

📖 解説: [`../04_multivariable.md`](../04_multivariable.md)

## このノートで触るもの
1. 等高線・3D グラフで多変数関数を見る
2. SymPy / 数値 / JAX で偏微分
3. 勾配ベクトルが等高線に垂直なのを可視化
4. 線形回帰のパラメータ偏微分

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (04_multivariable.md)](../04_multivariable.md)

In [ ]:
import numpy as np
import sympy as sp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, FloatSlider

%matplotlib inline

## 1. 多変数関数を 3D と等高線で見る

f(x, y) = x² + y² (お椀型)

$$
f(x, y) = x^2 + y^2 \quad \text{(お椀型)}
$$

In [ ]:
x = np.linspace(-3, 3, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = X**2 + Y**2

fig = plt.figure(figsize=(14, 5))

# 3D グラフ
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
ax1.set_title('f(x,y) = x² + y² (お椀)')

# 等高線
ax2 = fig.add_subplot(122)
cs = ax2.contour(X, Y, Z, levels=15, cmap='viridis')
ax2.clabel(cs, inline=True, fontsize=8)
ax2.set_xlabel('x'); ax2.set_ylabel('y')
ax2.set_title('等高線 (上から見たもの)')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. SymPy / 数値 / JAX で偏微分

$$
f(x, y) = x^2 + 3xy + y^2 \implies \frac{\partial f}{\partial x} = 2x + 3y, \quad \frac{\partial f}{\partial y} = 3x + 2y
$$

In [ ]:
# SymPy
x, y = sp.symbols('x y')
f = x**2 + 3*x*y + y**2
print('∂f/∂x =', sp.diff(f, x))
print('∂f/∂y =', sp.diff(f, y))

x0, y0 = 1.0, 2.0
print(f'\n(x,y) = (1,2) における:')
print(f'  ∂f/∂x = {sp.diff(f, x).subs([(x, x0), (y, y0)])}')
print(f'  ∂f/∂y = {sp.diff(f, y).subs([(x, x0), (y, y0)])}')

In [ ]:
# 数値偏微分
def f_num(x, y):
    return x**2 + 3*x*y + y**2

def partial(f, x, y, idx, h=1e-5):
    """idx=0 なら ∂/∂x, idx=1 なら ∂/∂y."""
    if idx == 0:
        return (f(x + h, y) - f(x - h, y)) / (2 * h)
    else:
        return (f(x, y + h) - f(x, y - h)) / (2 * h)

print(f'数値: ∂f/∂x = {partial(f_num, 1.0, 2.0, 0):.6f}')
print(f'数値: ∂f/∂y = {partial(f_num, 1.0, 2.0, 1):.6f}')

In [ ]:
# JAX (本命)
def f_jax(p):
    return p[0]**2 + 3*p[0]*p[1] + p[1]**2

grad_f = jax.grad(f_jax)
p = jnp.array([1.0, 2.0])
print(f'JAX: ∇f = {grad_f(p)}  ← (∂f/∂x, ∂f/∂y) を一気に')

## 3. 勾配ベクトルが等高線に垂直なのを可視化

勾配 $\nabla f$ は「最も急に登る方向」を指す:

$$
\nabla f(x_0, y_0) \perp \{(x, y) : f(x, y) = f(x_0, y_0)\}
$$

In [ ]:
def plot_grad_field():
    """f(x,y) = x² + y² の等高線と勾配ベクトルを描く."""
    x = np.linspace(-3, 3, 30)
    y = np.linspace(-3, 3, 30)
    X, Y = np.meshgrid(x, y)
    Z = X**2 + Y**2
    
    # 勾配 (細かく取らずに間引く)
    x_sparse = np.linspace(-3, 3, 12)
    y_sparse = np.linspace(-3, 3, 12)
    Xs, Ys = np.meshgrid(x_sparse, y_sparse)
    Gx = 2 * Xs  # ∂f/∂x = 2x
    Gy = 2 * Ys  # ∂f/∂y = 2y
    
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.contour(X, Y, Z, levels=12, cmap='Blues', alpha=0.7)
    ax.quiver(Xs, Ys, Gx, Gy, color='red', alpha=0.7, scale=80)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title('青: 等高線、赤: 勾配ベクトル ∇f\n勾配は常に「最も急に登る方向」を指す = 等高線に垂直')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.show()

plot_grad_field()

## 4. 最小点の確認

f(x, y) = (x − 1)² + (y − 2)² の最小値は (1, 2)、そこで勾配 = 0 のはず

$$
f(x, y) = (x - 1)^2 + (y - 2)^2
$$

最小点 $(1, 2)$ で $\nabla f = (0, 0)$

In [ ]:
def f(p):
    return (p[0] - 1)**2 + (p[1] - 2)**2

grad_f = jax.grad(f)

print('最小点 (1, 2) での勾配:', grad_f(jnp.array([1.0, 2.0])))
print('別の点 (0, 0) での勾配:', grad_f(jnp.array([0.0, 0.0])))
print('別の点 (3, 4) での勾配:', grad_f(jnp.array([3.0, 4.0])))
print('\n→ 最小点でのみ勾配がゼロ。これが "極値の必要条件"')

## 5. ML 例: 線形回帰のパラメータ偏微分

$$
L(w, b) = (y - (w x + b))^2
$$

$$
\frac{\partial L}{\partial w} = -2x(y - wx - b), \quad \frac{\partial L}{\partial b} = -2(y - wx - b)
$$

In [ ]:
# モデル: y_pred = w·x + b
# 損失: (y - y_pred)²

def loss(params, x, y_true):
    y_pred = params['w'] * x + params['b']
    return (y_true - y_pred) ** 2

# 各パラメータの偏微分
grad_loss = jax.grad(loss)

params = {'w': jnp.array(0.5), 'b': jnp.array(0.1)}
x_data = jnp.array(2.0)
y_data = jnp.array(3.0)

grads = grad_loss(params, x_data, y_data)
print('損失:', float(loss(params, x_data, y_data)))
print('∂L/∂w:', float(grads['w']))
print('∂L/∂b:', float(grads['b']))
print('\n→ この勾配を使って w, b を更新するのが ML の本体')

## まとめ
- 多変数関数は等高線で可視化できる
- 偏微分は「1変数だけ動かす」微分
- JAX の `grad` は勾配ベクトル (全偏微分) を一気に返す
- 勾配ベクトルは等高線に垂直 = 最急上昇方向
- 最小点では勾配がゼロ

## 次へ
→ [`05_gradient_jacobian.ipynb`](05_gradient_jacobian.ipynb)  解説 → [`../05_gradient_jacobian.md`](../05_gradient_jacobian.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`03_integrals.ipynb`](03_integrals.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`05_gradient_jacobian.ipynb`](05_gradient_jacobian.ipynb) |